# Wan 2.2 I2V A14B — Local Open-Source Test

**Goal:** Test the open-source Wan 2.2 Image-to-Video model (14B active params, MoE) running locally via HuggingFace diffusers.

**Model:** [Wan-AI/Wan2.2-I2V-A14B-Diffusers](https://huggingface.co/Wan-AI/Wan2.2-I2V-A14B-Diffusers)

**Requirements:**
- GPU with sufficient VRAM (~80GB for full precision, less with CPU offloading)
- diffusers installed from source
- torch with CUDA support

## Install Dependencies

In [ ]:
!pip install git+https://github.com/huggingface/diffusers torch torchvision ftfy sentencepiece accelerate

## Setup

In [ ]:
import torch
import numpy as np
from pathlib import Path

from diffusers import WanImageToVideoPipeline
from diffusers.utils import export_to_video, load_image

# Output directory
OUTPUT_DIR = Path(".")

# Check GPU
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Load Model

The A14B model uses Mixture-of-Experts (27B total params, 14B active). It needs significant VRAM.

- **Full GPU:** `pipe.to("cuda")` — fastest, requires ~80GB VRAM
- **CPU offload:** `pipe.enable_model_cpu_offload()` — slower but works with less VRAM

In [ ]:
MODEL_ID = "Wan-AI/Wan2.2-I2V-A14B-Diffusers"

pipe = WanImageToVideoPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True)

pipe.enable_model_cpu_offload()
#pipe.enable_vae_slicing()
#pipe.enable_vae_tiling()

print("Model loaded with CPU offload + VAE slicing/tiling.")

## Configuration

In [ ]:
# --- Generation config ---

SOURCE_IMAGE = "../legoman.png"
OUTPUT_FILE = "wan22_i2v_local_output.mp4"

PROMPT = (
    "The lego figure slowly turns its head to the right and raises one arm, "
    "as if waving hello. Smooth, gentle motion."
)

NEGATIVE_PROMPT = (
    "Bright tones, overexposed, static, blurry details, subtitles, style, artwork, "
    "painting, still image, overall gray, worst quality, low quality, JPEG artifacts, "
    "ugly, incomplete, extra fingers, poorly drawn hands, poorly drawn face, deformed, "
    "disfigured, malformed limbs, fused fingers, still frame, messy background, "
    "three legs, many people in background, walking backwards"
)

# Resolution tier: 480P or 720P
RESOLUTION = "480P"  # Use "720P" for higher quality (slower, more VRAM)

NUM_FRAMES = 81          # ~5 seconds at 16fps
GUIDANCE_SCALE = 3.5
NUM_INFERENCE_STEPS = 40
FPS = 16
SEED = 0

## Prepare Input Image

The image dimensions must be aligned to the model's VAE and patch size constraints.

In [ ]:
image = load_image(SOURCE_IMAGE)
print(f"Original size: {image.size}")

# Calculate aligned dimensions
resolution_map = {"480P": 480 * 832, "720P": 720 * 1280}
max_area = resolution_map[RESOLUTION]

aspect_ratio = image.height / image.width
mod_value = pipe.vae_scale_factor_spatial * pipe.transformer.config.patch_size[1]

height = round(np.sqrt(max_area * aspect_ratio)) // mod_value * mod_value
width = round(np.sqrt(max_area / aspect_ratio)) // mod_value * mod_value

image = image.resize((width, height))
print(f"Resized to: {width}x{height} (mod {mod_value}, area {width*height})")

# Display the input image
from IPython.display import display
display(image)

## Generate Video

In [ ]:
generator = torch.Generator(device="cuda").manual_seed(SEED)

output = pipe(
    image=image,
    prompt=PROMPT,
    negative_prompt=NEGATIVE_PROMPT,
    height=height,
    width=width,
    num_frames=NUM_FRAMES,
    guidance_scale=GUIDANCE_SCALE,
    num_inference_steps=NUM_INFERENCE_STEPS,
    generator=generator,
).frames[0]

output_path = OUTPUT_DIR / OUTPUT_FILE
export_to_video(output, str(output_path), fps=FPS)
print(f"Video saved: {output_path}")

## Review Output

In [ ]:
from IPython.display import display, Video

display(Video(str(output_path), embed=True, mimetype="video/mp4", width=480))

## Frame Inspection

Compare the input image with the first and last generated frames.

In [ ]:
import cv2
import matplotlib.pyplot as plt

cap = cv2.VideoCapture(str(output_path))

# First frame
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
ret, first_frame = cap.read()
first_frame = cv2.cvtColor(first_frame, cv2.COLOR_BGR2RGB)

# Last frame
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.set(cv2.CAP_PROP_POS_FRAMES, total_frames - 1)
ret, last_frame = cap.read()
last_frame = cv2.cvtColor(last_frame, cv2.COLOR_BGR2RGB)
cap.release()

# Input image as array
import numpy as np
input_arr = np.array(image)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(input_arr)
axes[0].set_title("Input Image")
axes[0].axis("off")

axes[1].imshow(first_frame)
axes[1].set_title("First Generated Frame")
axes[1].axis("off")

axes[2].imshow(last_frame)
axes[2].set_title("Last Generated Frame")
axes[2].axis("off")

plt.suptitle(f"Wan 2.2 I2V A14B — {NUM_FRAMES} frames, {NUM_INFERENCE_STEPS} steps", fontsize=14)
plt.tight_layout()
plt.show()

print(f"Total frames: {total_frames}, Duration: {total_frames/FPS:.1f}s")